# 🤖 AI Intraday Scalping System — Google Colab
### NSE/BSE Markets · Angel One SmartAPI · Capital: ₹1,00,000

---

**Steps covered in this notebook:**
1. Install dependencies
2. Import modules & configure
3. Step 1: Stock Universe + Pre-market Filter
4. Step 2: AI Scoring Engine (7 parameters)
5. Step 3: Scalping Strategy Engine
6. Step 4: Paper Trade Simulation
7. Daily P&L Report
8. Multi-day Simulation & Performance Metrics
9. Angel One SmartAPI Live Mode (Optional)

> ⚠️ **Paper Trading Only** — No real orders are placed.

---
## Cell 1 — Install Dependencies

In [37]:
# Install all required packages
!pip install -q yfinance pandas numpy ta streamlit smartapi-python textblob plotly pyotp loguru tabulate colorama scipy scikit-learn python-dotenv
print('✅ All packages installed successfully!')

✅ All packages installed successfully!


---
## Cell 2 — Clone Repository & Import Modules

In [38]:
import subprocess
import sys
import os

# Clone the repository (skip if already present)
REPO_URL = 'https://github.com/harnepal-hub/ai-scalping-system.git'
REPO_DIR = '/content/ai-scalping-system'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print(f'✅ Repository cloned to {REPO_DIR}')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print(f'✅ Repository updated at {REPO_DIR}')

# Add paths
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
os.chdir(REPO_DIR)

# Create log directory
os.makedirs('logs', exist_ok=True)
os.makedirs('data', exist_ok=True)

print('\n📁 Working directory:', os.getcwd())

✅ Repository updated at /content/ai-scalping-system

📁 Working directory: /content/ai-scalping-system


In [39]:
# Import all system modules
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from datetime import datetime, date

# Project modules
from config.config import (
    CAPITAL, MAX_POSITIONS, CAPITAL_PER_STOCK,
    MAX_RISK_PER_TRADE, MAX_DAILY_LOSS,
    MORNING_START, MORNING_END,
    AFTERNOON_START, AFTERNOON_END,
    ALL_SYMBOLS, WEIGHTS
)

from src.step1_universe_filter import (
    get_universe, get_premarket_data,
    apply_premarket_filters, get_filtered_universe
)
from src.step2_scoring_engine import (
    calculate_all_scores, get_top5,
    normalize_score, display_score_table
)
from src.step3_strategy_engine import (
    generate_trade_setup, check_entry_signal,
    display_trade_card, calculate_atr
)
from src.step4_paper_trader import (
    PaperTrader, calculate_charges
)

print('✅ All modules imported successfully!')
print(f'\n📊 Configuration:')
print(f'   Capital          : ₹{CAPITAL:,.0f}')
print(f'   Max positions    : {MAX_POSITIONS}')
print(f'   Capital / stock  : ₹{CAPITAL_PER_STOCK:,.0f}')
print(f'   Max risk / trade : ₹{MAX_RISK_PER_TRADE:,.0f}')
print(f'   Morning session  : {MORNING_START} – {MORNING_END}')
print(f'   Afternoon session: {AFTERNOON_START} – {AFTERNOON_END}')
print(f'   Universe size    : {len(ALL_SYMBOLS)} stocks')

✅ All modules imported successfully!

📊 Configuration:
   Capital          : ₹100,000
   Max positions    : 5
   Capital / stock  : ₹20,000
   Max risk / trade : ₹1,000
   Morning session  : 09:15 – 10:45
   Afternoon session: 13:30 – 15:15
   Universe size    : 100 stocks


---
## Cell 3 — Step 1: Load Universe & Pre-market Filter

In [40]:
print('=' * 60)
print('STEP 1: STOCK UNIVERSE FILTER')
print('=' * 60)

# Get full universe
full_universe = get_universe()
print(f'\n📋 Full universe: {len(full_universe)} stocks')

# For Colab demo: use a smaller subset for faster execution
# Change to full_universe for production
DEMO_SUBSET = [
    'RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'ICICIBANK.NS',
    'SBIN.NS', 'BHARTIARTL.NS', 'AXISBANK.NS', 'WIPRO.NS', 'HCLTECH.NS',
    'TATAMOTORS.NS', 'TATASTEEL.NS', 'NTPC.NS', 'POWERGRID.NS', 'ONGC.NS',
    'BAJFINANCE.NS', 'MARUTI.NS', 'TITAN.NS', 'SUNPHARMA.NS', 'DRREDDY.NS',
]

print(f'\n🔍 Running pre-market filter on {len(DEMO_SUBSET)} stocks (demo subset)...')
print('   (This fetches 20 days of daily data via yfinance — may take ~30s)\n')

# Fetch daily data for filtering
market_data = get_premarket_data(DEMO_SUBSET, period='20d', interval='1d')
filter_df = apply_premarket_filters(market_data)

if not filter_df.empty:
    passed_df = filter_df[filter_df['passes_filter']]
    tradable_symbols = passed_df['symbol'].tolist()

    print(f'\n✅ Tradable stocks after filter: {len(tradable_symbols)}')
    print(passed_df[['symbol', 'last_price', 'avg_volume', 'pct_change']].to_string(index=False))
else:
    print('⚠️  Filter returned no data. Using full demo subset as fallback.')
    tradable_symbols = DEMO_SUBSET

2026-03-25 00:58:59.051 | INFO     | src.step1_universe_filter:get_premarket_data:119 - Fetching 1d data for 20 symbols via yfinance …


STEP 1: STOCK UNIVERSE FILTER

📋 Full universe: 100 stocks

🔍 Running pre-market filter on 20 stocks (demo subset)...
   (This fetches 20 days of daily data via yfinance — may take ~30s)



ERROR:yfinance:$TATAMOTORS.NS: possibly delisted; no price data found  (period=20d) (Yahoo error = "No data found, symbol may be delisted")
2026-03-25 00:59:00.695 | DEBUG    | src.step1_universe_filter:get_premarket_data:126 - TATAMOTORS.NS: empty data returned, skipping.
2026-03-25 00:59:02.172 | INFO     | src.step1_universe_filter:get_premarket_data:133 - Successfully fetched data for 19/20 symbols.
2026-03-25 00:59:02.227 | INFO     | src.step1_universe_filter:apply_premarket_filters:212 - Pre-market filter: 19/19 stocks passed (0 removed).



✅ Tradable stocks after filter: 19
       symbol  last_price  avg_volume  pct_change
  RELIANCE.NS     1411.80  19247926.0        0.28
       TCS.NS     2398.80   3689563.0        0.63
  HDFCBANK.NS      764.90  51020428.0        2.79
      INFY.NS     1278.30  13199117.0        1.71
 ICICIBANK.NS     1251.20  18313886.0        2.33
      SBIN.NS     1030.80  16802098.0       -0.11
BHARTIARTL.NS     1802.10  10293148.0        0.35
  AXISBANK.NS     1192.70   7340123.0        1.89
     WIPRO.NS      188.74  16974644.0        0.64
   HCLTECH.NS     1373.30   3525298.0        1.08
 TATASTEEL.NS      190.79  36612750.0        1.93
      NTPC.NS      375.55  14365968.0        0.85
 POWERGRID.NS      299.00  18225410.0       -1.03
      ONGC.NS      268.05  25651865.0        0.98
BAJFINANCE.NS      849.00   9695522.0        4.48
    MARUTI.NS    12464.00    597698.0        0.88
     TITAN.NS     3899.50    844189.0        1.20
 SUNPHARMA.NS     1753.30   2893036.0       -0.29
   DRREDDY.NS 

---
## Cell 4 — Step 2: AI Scoring Engine

In [41]:
print('=' * 60)
print('STEP 2: AI SCORING ENGINE')
print('=' * 60)

print('\n📊 Fetching 5-min intraday data for scoring...')
print('   (This may take 1-2 minutes for multiple stocks)\n')

# Fetch 5-min intraday data
intraday_data = get_premarket_data(tradable_symbols, period='5d', interval='5m')
print(f'✅ Fetched intraday data for {len(intraday_data)} stocks')

# Score morning session
print('\n🌅 Scoring MORNING session...')
morning_scores = calculate_all_scores(
    tradable_symbols,
    intraday_data=intraday_data,
    session='morning'
)

if not morning_scores.empty:
    display_score_table(morning_scores, session='morning')
    morning_top5 = get_top5(morning_scores)
    print(f'\n🏆 MORNING TOP 5:')
    print(morning_top5[['rank', 'symbol', 'direction', 'composite_score']].to_string(index=False))

# Score afternoon session
print('\n🌆 Scoring AFTERNOON session...')
afternoon_scores = calculate_all_scores(
    tradable_symbols,
    intraday_data=intraday_data,
    session='afternoon'
)

if not afternoon_scores.empty:
    display_score_table(afternoon_scores, session='afternoon')
    afternoon_top5 = get_top5(afternoon_scores)
    print(f'\n🏆 AFTERNOON TOP 5:')
    print(afternoon_top5[['rank', 'symbol', 'direction', 'composite_score']].to_string(index=False))

2026-03-25 00:59:02.332 | INFO     | src.step1_universe_filter:get_premarket_data:119 - Fetching 5m data for 19 symbols via yfinance …


STEP 2: AI SCORING ENGINE

📊 Fetching 5-min intraday data for scoring...
   (This may take 1-2 minutes for multiple stocks)



2026-03-25 00:59:05.863 | INFO     | src.step1_universe_filter:get_premarket_data:133 - Successfully fetched data for 19/19 symbols.


✅ Fetched intraday data for 19 stocks

🌅 Scoring MORNING session...

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  AI SCORING ENGINE — TOP 5 PICKS (MORNING SESSION)
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
   #  Symbol            Dir  VolSurge   ATR   Mom  VWAP   ORB  News Trend  COMPOSITE
  ──────────────────────────────────────────────────────────────────────────────────────────────────────────
   1  RELIANCE        ▲ LONG      1.00  0.12  1.00  0.59  0.64  0.50  0.51     0.6777
   2  HCLTECH         ▲ LONG      0.89  0.08  0.80  0.88  0.77  0.50  0.40     0.6677
   3  INFY            ▲ LONG      0.44  0.10  0.80  0.93  0.85  0.50  0.35     0.5957
   4  ICICIBANK       ▲ LONG      1.00  0.12  0.66  0.49  0.44  0.50  0.59     0.5699
   5  POWERGRID       ▼ SHORT      0.84  0.16  0.80  0.44  0.41  0.50  0.45     0.5523
════════════════════════════════

---
## Cell 5 — Step 3: Strategy Engine — Trade Cards

In [42]:
print('=' * 60)
print('STEP 3: SCALPING STRATEGY ENGINE')
print('=' * 60)

def generate_cards_for_top5(top5_df, session_name, intraday_data):
    """Generate and display trade cards for the top 5 scored stocks."""
    trade_setups = []

    for _, row in top5_df.iterrows():
        sym = row['symbol']
        direction = row['direction']

        # Get latest price and ATR
        df = intraday_data.get(sym)
        if df is None or df.empty:
            print(f'  ⚠️  No data for {sym}, skipping.')
            continue

        last_price = float(df['Close'].dropna().iloc[-1])
        atr = calculate_atr(df, period=14)

        # Check entry signal
        is_valid, reason = check_entry_signal(df, session=session_name, direction=direction)
        signal_icon = '✅' if is_valid else '⚠️'
        print(f'  {signal_icon} {sym.replace(".NS", ""):<15} Signal: {reason}')

        # Generate setup (regardless of signal for demo purposes)
        setup = generate_trade_setup(
            sym, last_price, atr, direction, session_name
        )
        setup['composite_score'] = row.get('composite_score', 0)
        trade_setups.append(setup)
        display_trade_card(setup)

    return trade_setups

# Morning trade cards
print('\n🌅 MORNING SESSION — TRADE SETUPS')
print('-' * 50)
morning_setups = []
if 'morning_top5' in dir() and not morning_top5.empty:
    morning_setups = generate_cards_for_top5(morning_top5, 'morning', intraday_data)

# Afternoon trade cards
print('\n🌆 AFTERNOON SESSION — TRADE SETUPS')
print('-' * 50)
afternoon_setups = []
if 'afternoon_top5' in dir() and not afternoon_top5.empty:
    afternoon_setups = generate_cards_for_top5(afternoon_top5, 'afternoon', intraday_data)

2026-03-25 00:59:13.292 | INFO     | src.step3_strategy_engine:generate_trade_setup:165 - Trade setup generated: RELIANCE.NS LONG Entry=1414.00 SL=1408.34 TP1=1417.77 TP2=1423.43 Qty=14
2026-03-25 00:59:13.298 | INFO     | src.step3_strategy_engine:generate_trade_setup:165 - Trade setup generated: HCLTECH.NS LONG Entry=1373.30 SL=1368.94 TP1=1376.21 TP2=1380.57 Qty=14
2026-03-25 00:59:13.304 | INFO     | src.step3_strategy_engine:generate_trade_setup:165 - Trade setup generated: INFY.NS LONG Entry=1280.10 SL=1275.62 TP1=1283.09 TP2=1287.56 Qty=15
2026-03-25 00:59:13.311 | INFO     | src.step3_strategy_engine:generate_trade_setup:165 - Trade setup generated: ICICIBANK.NS LONG Entry=1251.20 SL=1246.06 TP1=1254.63 TP2=1259.77 Qty=15
2026-03-25 00:59:13.317 | INFO     | src.step3_strategy_engine:generate_trade_setup:165 - Trade setup generated: POWERGRID.NS SHORT Entry=298.40 SL=299.82 TP1=297.45 TP2=296.03 Qty=67
2026-03-25 00:59:13.324 | INFO     | src.step3_strategy_engine:generate_trad

STEP 3: SCALPING STRATEGY ENGINE

🌅 MORNING SESSION — TRADE SETUPS
--------------------------------------------------
  ⚠️ RELIANCE        Signal: Volume 422,127 < 1.5× avg 773,241

╔============================================╗
║  TRADE SETUP -- RELIANCE [LONG]            ║
╠============================================╣
║  Entry Price     : Rs   1,414.00           ║
║  Stop Loss       : Rs   1,408.34  (-0.40%) ║
║  Take Profit 1   : Rs   1,417.77  (+0.27%) ║
║  Take Profit 2   : Rs   1,423.43  (+0.67%) ║
║  Quantity        : 14 shares               ║
║  Max Loss        : Rs      79.20           ║
║  Risk-Reward     : 1 : 1.67                ║
║  Session         : MORNING                 ║
╚============================================╝

  ⚠️ HCLTECH         Signal: EMA(8) 1373.92 not above EMA(21) 1376.64

╔============================================╗
║  TRADE SETUP -- HCLTECH [LONG]             ║
╠============================================╣
║  Entry Price     : Rs   1,373.30       

---
## Cell 6 — Step 4: Paper Trade Simulation

In [63]:
print('=' * 60)
print('STEP 4: PAPER TRADE SIMULATION')
print('=' * 60)

import inspect

# ── Detect which API is loaded in memory ─────────────────��───────────────────
_sig = inspect.signature(trader.open_trade if 'trader' in dir() else PaperTrader().open_trade)
_params = list(_sig.parameters.keys())
_uses_tp1_tp2 = 'tp1' in _params   # old API
_uses_tp       = 'tp'  in _params and 'tp1' not in _params  # new API

print(f"  Detected PaperTrader API: {'tp1/tp2 (old)' if _uses_tp1_tp2 else 'tp (new)'}")

# ── Helper: extract TP values from a setup dict ───────────────────────────────
def _get_tp1(s):
    return s.get('tp1') or s.get('tp')

def _get_tp2(s):
    return s.get('tp2') or s.get('tp')

# ── Initialise trader ─────────────────────────────────────────────────────────
trader = PaperTrader(capital=CAPITAL)

# ── Open trades based on generated setups ─────────────────────────────────────
all_setups = morning_setups + afternoon_setups

if not all_setups:
    print('\n⚠️  No trade setups from live data. Using hardcoded demo trades...\n')

    demo_trades = [
        ('RELIANCE.NS', 'LONG',  2450.00, 2435.25, 2453.5, 2458.75, 8,  'morning'),
        ('HDFCBANK.NS', 'LONG',  1680.00, 1666.60, 1682.0, 1685.0,  11, 'morning'),
        ('TCS.NS',      'LONG',  3890.00, 3867.50, 3893.0, 3898.0,  5,  'afternoon'),
    ]
    for sym, dir_, entry, sl, tp1, tp2, qty, sess in demo_trades:
        if _uses_tp1_tp2:
            trader.open_trade(sym, dir_, entry, sl, tp1, tp2, qty, sess)
        else:
            trader.open_trade(sym, dir_, entry, sl, tp2, qty, sess)

    # Simulate exits
    for sym, price in [('RELIANCE.NS', 2459.0), ('HDFCBANK.NS', 1664.0), ('TCS.NS', 3898.0)]:
        trader.update_prices(sym, price)

else:
    print(f'\n📥 Opening {len(all_setups)} paper trades...\n')
    for setup in all_setups:
        tp1_price = _get_tp1(setup)
        tp2_price = _get_tp2(setup)
        sym_display = setup['stock'].replace('.NS', '')

        if _uses_tp1_tp2:
            success = trader.open_trade(
                stock           = setup['stock'],
                direction       = setup['direction'],
                entry           = setup['entry'],
                sl              = setup['sl'],
                tp1             = tp1_price,
                tp2             = tp2_price,
                qty             = setup['qty'],
                session         = setup['session'],
                composite_score = setup.get('composite_score', 0.0),
                atr             = setup.get('atr', 0.0),
                rr_ratio        = setup.get('rr_ratio', 0.0),
            )
        else:
            success = trader.open_trade(
                stock           = setup['stock'],
                direction       = setup['direction'],
                entry           = setup['entry'],
                sl              = setup['sl'],
                tp              = tp2_price,
                qty             = setup['qty'],
                session         = setup['session'],
                composite_score = setup.get('composite_score', 0.0),
                atr             = setup.get('atr', 0.0),
                rr_ratio        = setup.get('rr_ratio', 0.0),
            )

        icon = '✅' if success else '⏭️ skipped'
        print(f'  {icon} {sym_display} {setup["direction"]} @ ₹{setup["entry"]:.2f}  '
              f'SL=₹{setup["sl"]:.2f}  TP1=₹{tp1_price:.2f}  TP2=₹{tp2_price:.2f}  Qty={setup["qty"]}')

    print('\n  ⏰ EOD square-off...')
    trader.eod_squareoff()

print(f'\n✅ Simulation complete. Total closed trades: {len(trader.closed_trades)}')

2026-03-25 01:15:24.731 | INFO     | src.step4_paper_trader:open_trade:226 - [OPEN] RELIANCE.NS LONG @ ₹1414.00  SL=1408.34  TP1=1417.77  TP2=1423.43  Qty=14
2026-03-25 01:15:24.734 | INFO     | src.step4_paper_trader:open_trade:226 - [OPEN] HCLTECH.NS LONG @ ₹1373.30  SL=1368.94  TP1=1376.21  TP2=1380.57  Qty=14
2026-03-25 01:15:24.738 | INFO     | src.step4_paper_trader:open_trade:226 - [OPEN] INFY.NS LONG @ ₹1280.10  SL=1275.62  TP1=1283.09  TP2=1287.56  Qty=15
2026-03-25 01:15:24.739 | INFO     | src.step4_paper_trader:open_trade:226 - [OPEN] ICICIBANK.NS LONG @ ₹1251.20  SL=1246.06  TP1=1254.63  TP2=1259.77  Qty=15
2026-03-25 01:15:24.741 | INFO     | src.step4_paper_trader:open_trade:226 - [OPEN] POWERGRID.NS SHORT @ ₹298.40  SL=299.82  TP1=297.45  TP2=296.03  Qty=67
2026-03-25 01:15:24.745 | WARNING  | src.step4_paper_trader:open_trade:192 - Trade for RELIANCE.NS already open. Skipping.
2026-03-25 01:15:24.747 | WARNING  | src.step4_paper_trader:open_trade:192 - Trade for HCLTEC

STEP 4: PAPER TRADE SIMULATION
  Detected PaperTrader API: tp1/tp2 (old)

📥 Opening 10 paper trades...

  ✅ RELIANCE LONG @ ₹1414.00  SL=₹1408.34  TP1=₹1417.77  TP2=₹1423.43  Qty=14
  ✅ HCLTECH LONG @ ₹1373.30  SL=₹1368.94  TP1=₹1376.21  TP2=₹1380.57  Qty=14
  ✅ INFY LONG @ ₹1280.10  SL=₹1275.62  TP1=₹1283.09  TP2=₹1287.56  Qty=15
  ✅ ICICIBANK LONG @ ₹1251.20  SL=₹1246.06  TP1=₹1254.63  TP2=₹1259.77  Qty=15
  ✅ POWERGRID SHORT @ ₹298.40  SL=₹299.82  TP1=₹297.45  TP2=₹296.03  Qty=67
  ⏭️ skipped RELIANCE LONG @ ₹1414.00  SL=₹1408.34  TP1=₹1417.77  TP2=₹1423.43  Qty=14
  ⏭️ skipped HCLTECH LONG @ ₹1373.30  SL=₹1368.94  TP1=₹1376.21  TP2=₹1380.57  Qty=14
  ⏭️ skipped ICICIBANK LONG @ ₹1251.20  SL=₹1246.06  TP1=₹1254.63  TP2=₹1259.77  Qty=15
  ⏭️ skipped INFY LONG @ ₹1280.10  SL=₹1275.62  TP1=₹1283.09  TP2=₹1287.56  Qty=15
  ✅ DRREDDY LONG @ ₹1259.60  SL=₹1253.48  TP1=₹1263.68  TP2=₹1269.80  Qty=15

  ⏰ EOD square-off...

✅ Simulation complete. Total closed trades: 6


---
## Cell 7 — Daily P&L Report

In [64]:
import plotly.graph_objects as go
from IPython.display import display

# Print formatted daily report
trader.print_daily_report()

# Save to CSV
TRADE_CSV = 'data/paper_trades.csv'
trader.save_to_csv(TRADE_CSV)
print(f'💾 Trades saved to {TRADE_CSV}')

# Plotly cumulative P&L chart
summary = trader.get_daily_summary()
if summary['trades']:
    trades_df = pd.DataFrame(summary['trades'])
    trades_df['cumulative_pnl'] = trades_df['net_pnl'].cumsum()

    fig = go.Figure()

    # Bar chart for individual trades
    colors = ['#00c853' if v >= 0 else '#d32f2f' for v in trades_df['net_pnl']]
    fig.add_trace(go.Bar(
        x=[t.replace('.NS', '') for t in trades_df['stock']],
        y=trades_df['net_pnl'],
        name='Net P&L per Trade',
        marker_color=colors,
        text=[f'₹{v:+.0f}' for v in trades_df['net_pnl']],
        textposition='outside',
    ))

    fig.update_layout(
        title='📊 Paper Trade Results — Net P&L per Trade',
        xaxis_title='Stock',
        yaxis_title='Net P&L (₹)',
        height=400,
        showlegend=False,
    )
    fig.show()

    # Cumulative P&L line
    fig2 = go.Figure(go.Scatter(
        x=list(range(len(trades_df))),
        y=CAPITAL + trades_df['cumulative_pnl'],
        mode='lines+markers',
        line=dict(color='#1976d2', width=2),
        fill='tozeroy',
        fillcolor='rgba(25, 118, 210, 0.1)',
        name='Capital',
    ))
    fig2.add_hline(y=CAPITAL, line_dash='dash', line_color='gray', annotation_text='Initial Capital')
    fig2.update_layout(
        title='📈 Cumulative Capital Curve',
        xaxis_title='Trade #',
        yaxis_title='Capital (₹)',
        height=400,
    )
    fig2.show()
else:
    print('No trades to chart.')

2026-03-25 01:15:31.932 | INFO     | src.step4_paper_trader:save_to_csv:527 - Saved 6 trades to data/paper_trades.csv



═══════════════════════════════════════════════════════════════════════════════════════════════
  📅 PAPER TRADE REPORT — 2026-03-25
═══════════════════════════════════════════════════════════════════════════════════════════════
  SESSION    │ STOCK      │ DIR   │   ENTRY │    EXIT │  QTY │    GROSS │   FEES │  NET P&L │ RESULT
  ───────────────────────────────────────────────────────────────────────────────────────────
  MORNING    │ RELIANCE   │ LONG  │ ₹1414.0 │ ₹1417.8 │   14 │ +₹    53 │ -₹   54 │ -₹      2 │ ❌ LOSS
  MORNING    │ HCLTECH    │ LONG  │ ₹1373.3 │ ₹1376.2 │   14 │ +₹    41 │ -₹   54 │ -₹     13 │ ❌ LOSS
  MORNING    │ INFY       │ LONG  │ ₹1280.1 │ ₹1283.1 │   15 │ +₹    45 │ -₹   54 │ -₹      9 │ ❌ LOSS
  MORNING    │ ICICIBANK  │ LONG  │ ₹1251.2 │ ₹1254.6 │   15 │ +₹    51 │ -₹   54 │ -₹      3 │ ❌ LOSS
  MORNING    │ POWERGRID  │ SHORT │ ₹ 298.4 │ ₹ 297.4 │   67 │ +₹    64 │ -₹   54 │ +₹      9 │ ✅ WIN
  AFTERNOON  │ DRREDDY    │ LONG  │ ₹1259.6 │ ₹1263.7 │   15 │

---
## Cell 8 — Multi-Day Simulation & Performance Metrics

In [65]:
print('=' * 60)
print('MULTI-DAY SIMULATION (5 trading days — sample data)')
print('=' * 60)

# Load sample trade history (ships with the repo)
sample_csv = 'data/sample_trades.csv'

multi_trader = PaperTrader(capital=CAPITAL)
multi_trader.load_history(sample_csv)

if not multi_trader.closed_trades:
    print('\n⚠️  Sample trades not found. Generating synthetic data...')
    import random
    random.seed(42)
    np.random.seed(42)

    stocks = ['RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'SBIN.NS']
    prices = [2450, 3890, 1680, 1590, 812]

    from datetime import timedelta
    base_date = date(2026, 3, 18)

    cap = float(CAPITAL)
    for day_offset in range(5):
        trade_date = base_date + timedelta(days=day_offset)
        for i, (sym, price) in enumerate(zip(stocks, prices)):
            entry = price * (1 + np.random.uniform(-0.005, 0.005))
            move = np.random.choice([-1, 1]) * np.random.uniform(0.003, 0.012) * entry
            exit_price = entry + move
            qty = max(1, int(20000 / entry))
            gross = (exit_price - entry) * qty
            fees = calculate_charges(min(entry, exit_price), max(entry, exit_price), qty)
            net = gross - fees
            cap += net

            multi_trader.closed_trades.append({
                'date': str(trade_date),
                'session': 'morning' if i < 3 else 'afternoon',
                'stock': sym,
                'symbol_token': '',
                'direction': 'LONG',
                'entry_price': round(entry, 2),
                'exit_price': round(exit_price, 2),
                'qty': qty,
                'sl': round(entry * 0.994, 2),
                'tp': round(entry * 1.005, 2),
                'exit_reason': 'TP' if gross > 0 else 'SL',
                'gross_pnl': round(gross, 2),
                'fees': round(fees, 2),
                'net_pnl': round(net, 2),
                'entry_time': f'{trade_date} 09:30:00',
                'exit_time': f'{trade_date} 10:15:00',
                'composite_score': round(np.random.uniform(0.55, 0.85), 4),
                'atr': round(price * 0.004, 2),
                'rr_ratio': 2.3,
                'capital_after': round(cap, 2),
            })
    multi_trader.capital = cap

all_trades_df = pd.DataFrame(multi_trader.closed_trades)
all_trades_df['net_pnl'] = pd.to_numeric(all_trades_df['net_pnl'], errors='coerce').fillna(0)
all_trades_df['date'] = pd.to_datetime(all_trades_df['date'])

# ── Performance Metrics ─────────────────────────────────────────────────────
total = len(all_trades_df)
wins = (all_trades_df['net_pnl'] >= 0).sum()
win_rate = wins / total * 100
gross_profit = all_trades_df[all_trades_df['net_pnl'] > 0]['net_pnl'].sum()
gross_loss = abs(all_trades_df[all_trades_df['net_pnl'] < 0]['net_pnl'].sum())
profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')

daily_pnl = all_trades_df.groupby(all_trades_df['date'].dt.date)['net_pnl'].sum()
daily_ret = daily_pnl / CAPITAL
sharpe = (daily_ret.mean() / daily_ret.std() * (250**0.5)) if daily_ret.std() > 0 else 0
cum_cap = CAPITAL + daily_pnl.cumsum()
peak = cum_cap.cummax()
max_dd = ((cum_cap - peak) / peak * 100).min()

print(f'\n{'='*50}')
print(f'  MULTI-DAY PERFORMANCE SUMMARY')
print(f'{'='*50}')
print(f'  Total trades    : {total}')
print(f'  Wins            : {wins} ({win_rate:.1f}%)')
print(f'  Losses          : {total - wins}')
print(f'  Profit Factor   : {profit_factor:.2f}')
print(f'  Sharpe Ratio    : {sharpe:.2f} (annualised)')
print(f'  Max Drawdown    : {max_dd:.2f}%')
print(f'  Net P&L         : ₹{all_trades_df["net_pnl"].sum():+,.2f}')
print(f'  Final Capital   : ₹{multi_trader.capital:,.2f}')
print(f'  Total Return    : {(multi_trader.capital - CAPITAL)/CAPITAL*100:+.2f}%')
print(f'{'='*50}')

# ── Charts ─────────────────────────────────────────────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Daily P&L', 'Cumulative Capital'),
    vertical_spacing=0.15
)

# Daily P&L bars
colors = ['#00c853' if v >= 0 else '#d32f2f' for v in daily_pnl]
fig.add_trace(go.Bar(
    x=[str(d) for d in daily_pnl.index],
    y=daily_pnl.values,
    marker_color=colors,
    name='Daily Net P&L',
    text=[f'₹{v:+,.0f}' for v in daily_pnl.values],
    textposition='outside',
), row=1, col=1)

# Capital curve
fig.add_trace(go.Scatter(
    x=[str(d) for d in cum_cap.index],
    y=cum_cap.values,
    mode='lines+markers',
    line=dict(color='#1976d2', width=2),
    fill='tozeroy',
    fillcolor='rgba(25, 118, 210, 0.1)',
    name='Capital',
), row=2, col=1)

fig.add_hline(y=CAPITAL, line_dash='dash', line_color='gray', row=2, col=1)
fig.update_layout(title='📈 Multi-Day Paper Trading Performance', height=700, showlegend=False)
fig.show()

2026-03-25 01:15:38.116 | INFO     | src.step4_paper_trader:load_history:549 - Loaded 10 trades from data/sample_trades.csv


MULTI-DAY SIMULATION (5 trading days — sample data)

  MULTI-DAY PERFORMANCE SUMMARY
  Total trades    : 10
  Wins            : 7 (70.0%)
  Losses          : 3
  Profit Factor   : 1.66
  Sharpe Ratio    : 20.31 (annualised)
  Max Drawdown    : 0.00%
  Net P&L         : ₹+421.96
  Final Capital   : ₹100,421.97
  Total Return    : +0.42%


---
## Cell 9 — Angel One SmartAPI Live Mode (Optional)

This cell shows how to connect to the Angel One API for live data.
**Paper trading only — no real orders are placed.**

### Setup Instructions
1. Get your API key from [Angel One SmartAPI](https://smartapi.angelbroking.com/)
2. Enable TOTP in your Angel One account
3. Add credentials to Colab Secrets (left sidebar → 🔑 icon):
   - `ANGEL_ONE_API_KEY`
   - `ANGEL_ONE_CLIENT_ID`
   - `ANGEL_ONE_PASSWORD`
   - `ANGEL_ONE_TOTP_SECRET`

In [56]:
# ── Angel One SmartAPI Live Mode ────────────────────────────────────────────
# Set USE_LIVE_DATA = True to enable (requires valid API credentials)
USE_LIVE_DATA = False

if USE_LIVE_DATA:
    try:
        from google.colab import userdata
        import pyotp
        from SmartApi import SmartConnect

        API_KEY       = userdata.get('CU4rg3Qn')
        CLIENT_ID     = userdata.get('AIIRA36917')
        PASSWORD      = userdata.get('2244')
        TOTP_SECRET   = userdata.get('Y56T4B5DTWJNQOMOJUAKLPGAX4')

        # Generate TOTP
        totp = pyotp.TOTP(TOTP_SECRET).now()

        # Connect
        smartapi = SmartConnect(api_key=API_KEY)
        session_data = smartapi.generateSession(CLIENT_ID, PASSWORD, totp)

        if session_data.get('status'):
            auth_token = session_data['data']['jwtToken']
            refresh_token = session_data['data']['refreshToken']
            print(f'✅ Connected to Angel One SmartAPI')
            print(f'   Client ID: {CLIENT_ID}')

            # Example: Fetch live LTP for RELIANCE
            # ltp_data = smartapi.ltpData('NSE', 'RELIANCE-EQ', '2885')
            # print(f'   RELIANCE LTP: ₹{ltp_data["data"]["ltp"]}')

            print('\n💡 Angel One is connected. You can now:')
            print('   • Fetch live prices: smartapi.ltpData(exchange, symbol, token)')
            print('   • Stream real-time: use SmartWebSocket for tick data')
            print('   • All paper trades still use PaperTrader (no real orders)')
        else:
            print('❌ Angel One login failed:', session_data.get('message'))

    except ImportError:
        print('❌ SmartAPI not installed. Run: !pip install smartapi-python')
    except Exception as e:
        print(f'❌ Angel One connection error: {e}')
        print('   Check your credentials in Colab Secrets.')
else:
    print('ℹ️  Angel One live mode is DISABLED (USE_LIVE_DATA = False).')
    print('   Using yfinance for all data. Set USE_LIVE_DATA = True to enable.')
    print('\n   Required Colab Secrets:')
    print('   • ANGEL_ONE_API_KEY')
    print('   • ANGEL_ONE_CLIENT_ID')
    print('   • ANGEL_ONE_PASSWORD')
    print('   • ANGEL_ONE_TOTP_SECRET')

ℹ️  Angel One live mode is DISABLED (USE_LIVE_DATA = False).
   Using yfinance for all data. Set USE_LIVE_DATA = True to enable.

   Required Colab Secrets:
   • ANGEL_ONE_API_KEY
   • ANGEL_ONE_CLIENT_ID
   • ANGEL_ONE_PASSWORD
   • ANGEL_ONE_TOTP_SECRET


---
## 🎯 Summary

| Step | Module | Status |
|------|--------|--------|
| 1 | Universe Filter | ✅ Complete |
| 2 | AI Scoring Engine | ✅ Complete |
| 3 | Strategy Engine | ✅ Complete |
| 4 | Paper Trader + P&L | ✅ Complete |
| 5 | ML Model (coming) | 🔄 Planned |
| 6 | Live Execution (coming) | 🔄 Planned |

### ▶️ Run the Streamlit Dashboard
```bash
!streamlit run /content/ai-scalping-system/src/dashboard.py &
# Open the ngrok URL shown in output
```

---
> ⚠️ **Disclaimer:** This is a paper trading simulation. No real trades are executed. Not financial advice.

In [68]:
LIVE_CSV = 'data/paper_trades_journal.csv'   # ← match the name already on your disk

In [69]:
# ═══════════════════════════════════════════════════════════════
# CELL 10 — DAILY LIVE SCORING + AUTO CSV SAVE & WEEKLY RETRIEVE
# Run once every trading day. Data accumulates in live_trades.csv
# ═══════════════════════════════════════════════════════════════

import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, date, timedelta
from pathlib import Path

warnings.filterwarnings('ignore')
sys.path.insert(0, '/content/ai-scalping-system')
os.chdir('/content/ai-scalping-system')
os.makedirs('data', exist_ok=True)

from src.step1_universe_filter import get_premarket_data, apply_premarket_filters
from src.step2_scoring_engine   import calculate_all_scores, get_top5
from src.step3_strategy_engine  import generate_trade_setup, calculate_atr, display_trade_card
from src.step4_paper_trader     import PaperTrader, calculate_charges, TRADE_CSV_COLUMNS
from config.config              import CAPITAL

# ── CONFIG ────────────────────────────────────────────────────────
LIVE_CSV = 'data/live_trades.csv'   # master file — appended every day
TODAY    = str(date.today())

SCAN_SYMBOLS = [
    'RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'ICICIBANK.NS',
    'SBIN.NS', 'BHARTIARTL.NS', 'AXISBANK.NS', 'WIPRO.NS', 'HCLTECH.NS',
    'TATAMOTORS.NS', 'TATASTEEL.NS', 'NTPC.NS', 'POWERGRID.NS', 'ONGC.NS',
    'BAJFINANCE.NS', 'MARUTI.NS', 'TITAN.NS', 'SUNPHARMA.NS', 'DRREDDY.NS',
]

print(f'📅 DAILY LIVE SCAN — {TODAY}\n')

# ── STEP 1: Fetch & Filter ─────────────────────────────────────────
print('🔍 Step 1: Fetching market data & applying filters...')
daily_data = get_premarket_data(SCAN_SYMBOLS, period='20d', interval='1d')
filter_df  = apply_premarket_filters(daily_data)

if not filter_df.empty and 'passes_filter' in filter_df.columns:
    tradable = filter_df[filter_df['passes_filter']]['symbol'].tolist()
else:
    tradable = SCAN_SYMBOLS
print(f'   ✅ {len(tradable)} stocks passed filter\n')

# ── STEP 2: Score stocks ──────────────────────────────────────────
print('📊 Step 2: Fetching 5-min data & scoring...')
intraday = get_premarket_data(tradable, period='5d', interval='5m')

morning_scores   = calculate_all_scores(tradable, intraday_data=intraday, session='morning')
afternoon_scores = calculate_all_scores(tradable, intraday_data=intraday, session='afternoon')

top5_m = get_top5(morning_scores)   if not morning_scores.empty   else pd.DataFrame()
top5_a = get_top5(afternoon_scores) if not afternoon_scores.empty else pd.DataFrame()
print(f'   🌅 Morning top 5   : {", ".join(top5_m["symbol"].str.replace(".NS","").tolist()) if not top5_m.empty else "none"}')
print(f'   🌆 Afternoon top 5 : {", ".join(top5_a["symbol"].str.replace(".NS","").tolist()) if not top5_a.empty else "none"}\n')

# ── STEP 3: Generate Trade Setups ─────────────────────────────────
print('📋 Step 3: Generating trade setups...')

def build_setups(top5_df, session_name):
    setups = []
    if top5_df.empty:
        return setups
    for _, row in top5_df.iterrows():
        sym = row['symbol']
        df  = intraday.get(sym)
        if df is None or df.empty:
            continue
        price = float(df['Close'].dropna().iloc[-1])
        atr   = calculate_atr(df, period=14)
        setup = generate_trade_setup(sym, price, atr, row['direction'], session_name)
        setup['composite_score'] = row.get('composite_score', 0.0)
        display_trade_card(setup)
        setups.append(setup)
    return setups

all_setups = build_setups(top5_m, 'morning') + build_setups(top5_a, 'afternoon')

# ── STEP 4: Paper Trade Simulation ────────────────────────────────
print(f'\n💼 Step 4: Paper trading {len(all_setups)} setups...')

# Load history to carry forward capital
daily_trader = PaperTrader(capital=CAPITAL)
if Path(LIVE_CSV).exists():
    daily_trader.load_history(LIVE_CSV)

# Check if today already saved
already_saved = any(t['date'] == TODAY for t in daily_trader.closed_trades)

if already_saved:
    print(f'   ℹ️  Today ({TODAY}) already saved. Skipping simulation.')
elif all_setups:
    day_trader = PaperTrader(capital=daily_trader.capital)

    for s in all_setups:
        day_trader.open_trade(
            stock=s['stock'], direction=s['direction'],
            entry=s['entry'], sl=s['sl'], tp=s['tp'],
            qty=s['qty'], session=s['session'],
            composite_score=s.get('composite_score', 0),
            atr=s.get('atr', 0), rr_ratio=s.get('rr_ratio', 0),
        )

    # Simulate price hits from last 20 candles
    for s in all_setups:
        df = intraday.get(s['stock'])
        if df is None or df.empty:
            continue
        for _, candle in df.tail(20).iterrows():
            day_trader.update_prices(s['stock'], float(candle['High']))
            day_trader.update_prices(s['stock'], float(candle['Low']))
            day_trader.update_prices(s['stock'], float(candle['Close']))

    day_trader.eod_squareoff()
    day_trader.print_daily_report()

    # Append today's trades to master CSV
    if day_trader.closed_trades:
        today_df = pd.DataFrame(day_trader.closed_trades)
        for col in TRADE_CSV_COLUMNS:
            if col not in today_df.columns:
                today_df[col] = None
        today_df = today_df[TRADE_CSV_COLUMNS]

        if Path(LIVE_CSV).exists():
            master_df = pd.concat([pd.read_csv(LIVE_CSV), today_df], ignore_index=True)
        else:
            master_df = today_df

        master_df.to_csv(LIVE_CSV, index=False)
        print(f'\n💾 Saved {len(today_df)} trades → {LIVE_CSV} (total: {len(master_df)} rows)')
    else:
        print('\n⚠️  No trades closed today.')
else:
    print('\n⚠️  No setups generated.')

# ── STEP 5: Show full week history ────────────────────────────────
print('\n' + '═'*60)
print('  📂 WEEKLY HISTORY FROM live_trades.csv')
print('═'*60)

if not Path(LIVE_CSV).exists():
    print('  No data yet. Run this cell on a trading day first.')
else:
    hist = pd.read_csv(LIVE_CSV)
    hist['net_pnl'] = pd.to_numeric(hist['net_pnl'], errors='coerce').fillna(0)
    hist['date']    = pd.to_datetime(hist['date'])
    week_df = hist[hist['date'] >= pd.Timestamp(date.today() - timedelta(days=7))]

    print(f'\n  Trades this week : {len(week_df)}')
    print(f'  Net P&L (week)   : ₹{week_df["net_pnl"].sum():+,.2f}')
    wins = (week_df['net_pnl'] >= 0).sum()
    if len(week_df):
        print(f'  Win rate         : {wins}/{len(week_df)} = {wins/len(week_df)*100:.1f}%')

    # Daily breakdown
    print(f'\n  {"DATE":<12} {"TRADES":>6} {"WINS":>5} {"NET P&L":>10}')
    print('  ' + '─'*38)
    for day, grp in week_df.groupby(week_df['date'].dt.date):
        n = len(grp); w = (grp['net_pnl'] >= 0).sum(); net = grp['net_pnl'].sum()
        print(f'  {str(day):<12} {n:>6} {w:>5}  {"+" if net>=0 else ""}₹{abs(net):,.0f}')

    # Chart
    if len(week_df) >= 2:
        daily_pnl = week_df.groupby(week_df['date'].dt.date)['net_pnl'].sum()
        cum_cap   = CAPITAL + daily_pnl.cumsum()
        fig = make_subplots(rows=2, cols=1,
                            subplot_titles=('Daily Net P&L', 'Cumulative Capital'),
                            vertical_spacing=0.18)
        colors = ['#00c853' if v >= 0 else '#d32f2f' for v in daily_pnl]
        fig.add_trace(go.Bar(x=[str(d) for d in daily_pnl.index], y=daily_pnl.values,
                             marker_color=colors,
                             text=[f'₹{v:+,.0f}' for v in daily_pnl.values],
                             textposition='outside'), row=1, col=1)
        fig.add_trace(go.Scatter(x=[str(d) for d in cum_cap.index], y=cum_cap.values,
                                 mode='lines+markers',
                                 line=dict(color='#1976d2', width=2),
                                 fill='tozeroy',
                                 fillcolor='rgba(25,118,210,0.1)'), row=2, col=1)
        fig.add_hline(y=CAPITAL, line_dash='dash', line_color='gray', row=2, col=1)
        fig.update_layout(title='📈 Weekly Live Performance', height=700, showlegend=False)
        fig.show()

    # Auto-download CSV
    print('\n📥 Downloading live_trades.csv...')
    try:
        from google.colab import files
        files.download(LIVE_CSV)
        print('   ✅ Download started!')
    except Exception:
        print(f'   📂 File at: /content/ai-scalping-system/{LIVE_CSV}')

print('\n✅ Cell 10 complete!')

2026-03-25 01:16:31.450 | INFO     | src.step1_universe_filter:get_premarket_data:119 - Fetching 1d data for 20 symbols via yfinance …


📅 DAILY LIVE SCAN — 2026-03-25

🔍 Step 1: Fetching market data & applying filters...


ERROR:yfinance:$TATAMOTORS.NS: possibly delisted; no price data found  (period=20d) (Yahoo error = "No data found, symbol may be delisted")
2026-03-25 01:16:32.832 | DEBUG    | src.step1_universe_filter:get_premarket_data:126 - TATAMOTORS.NS: empty data returned, skipping.
2026-03-25 01:16:33.859 | INFO     | src.step1_universe_filter:get_premarket_data:133 - Successfully fetched data for 19/20 symbols.
2026-03-25 01:16:33.872 | INFO     | src.step1_universe_filter:apply_premarket_filters:212 - Pre-market filter: 19/19 stocks passed (0 removed).
2026-03-25 01:16:33.874 | INFO     | src.step1_universe_filter:get_premarket_data:119 - Fetching 5m data for 19 symbols via yfinance …


   ✅ 19 stocks passed filter

📊 Step 2: Fetching 5-min data & scoring...


2026-03-25 01:16:35.839 | INFO     | src.step1_universe_filter:get_premarket_data:133 - Successfully fetched data for 19/19 symbols.
2026-03-25 01:16:41.441 | INFO     | src.step3_strategy_engine:generate_trade_setup:165 - Trade setup generated: RELIANCE.NS LONG Entry=1414.00 SL=1408.34 TP1=1417.77 TP2=1423.43 Qty=14
2026-03-25 01:16:41.444 | INFO     | src.step3_strategy_engine:generate_trade_setup:165 - Trade setup generated: HCLTECH.NS LONG Entry=1373.30 SL=1368.94 TP1=1376.21 TP2=1380.57 Qty=14
2026-03-25 01:16:41.447 | INFO     | src.step3_strategy_engine:generate_trade_setup:165 - Trade setup generated: INFY.NS LONG Entry=1280.10 SL=1275.62 TP1=1283.09 TP2=1287.56 Qty=15
2026-03-25 01:16:41.450 | INFO     | src.step3_strategy_engine:generate_trade_setup:165 - Trade setup generated: ICICIBANK.NS LONG Entry=1251.20 SL=1246.06 TP1=1254.63 TP2=1259.77 Qty=15
2026-03-25 01:16:41.453 | INFO     | src.step3_strategy_engine:generate_trade_setup:165 - Trade setup generated: POWERGRID.NS S

   🌅 Morning top 5   : RELIANCE, HCLTECH, INFY, ICICIBANK, POWERGRID
   🌆 Afternoon top 5 : RELIANCE, HCLTECH, ICICIBANK, INFY, DRREDDY

📋 Step 3: Generating trade setups...

╔============================================╗
║  TRADE SETUP -- RELIANCE [LONG]            ║
╠============================================╣
║  Entry Price     : Rs   1,414.00           ║
║  Stop Loss       : Rs   1,408.34  (-0.40%) ║
║  Take Profit 1   : Rs   1,417.77  (+0.27%) ║
║  Take Profit 2   : Rs   1,423.43  (+0.67%) ║
║  Quantity        : 14 shares               ║
║  Max Loss        : Rs      79.20           ║
║  Risk-Reward     : 1 : 1.67                ║
║  Session         : MORNING                 ║
╚============================================╝


╔============================================╗
║  TRADE SETUP -- HCLTECH [LONG]             ║
╠============================================╣
║  Entry Price     : Rs   1,373.30           ║
║  Stop Loss       : Rs   1,368.94  (-0.32%) ║
║  Take Profit 1   : Rs 

KeyError: 'tp'